In [1]:
import requests
import pandas as pd

OTP_URL = "http://localhost:8080/otp/gtfs/v1"

def query_otp(query: str, variables: dict | None = None) -> dict:
    """Run a GraphQL query against the OTP server and return the `data` payload."""
    resp = requests.post(
        OTP_URL,
        json={"query": query, "variables": variables or {}},
        headers={"Content-Type": "application/json"},
        timeout=60,
    )
    resp.raise_for_status()
    payload = resp.json()
    if "errors" in payload:
        raise RuntimeError(payload["errors"])
    return payload["data"]

In [2]:
PLAN_QUERY = """
query Plan($from: InputCoordinates!, $to: InputCoordinates!, $date: String!, $time: String!) {
  plan(
    from: $from
    to: $to
    date: $date
    time: $time
    transportModes: [{ mode: TRANSIT }, { mode: WALK }]
  ) {
    itineraries {
      start
      end
      duration
      numberOfTransfers
      legs {
        mode
        distance
        duration
        start { scheduledTime }
        end { scheduledTime }
        route { shortName longName }
        from { name }
        to { name }
        fareProducts {
          product {
            name
            ... on DefaultFareProduct {
              price { amount currency { code } }
            }
          }
        }
      }
    }
  }
}
"""

variables = {
    "from": {"lat": -37.81, "lon": 144.96},  # Melbourne
    "to": {"lat": -37.872, "lon": 144.9769},    # Sydney-37.80371 144.9769
    "date": "2026-05-20",
    "time": "08:00",
}

data = query_otp(PLAN_QUERY, variables)
itineraries = data["plan"]["itineraries"]
print(f"{len(itineraries)} itineraries found")

3 itineraries found


In [3]:
def format_duration(seconds: float | None) -> str | None:
    """e.g. 5025 -> '1h 23m', 180 -> '3m'"""
    if seconds is None:
        return None
    total_min = round(seconds / 60)
    h, m = divmod(total_min, 60)
    return f"{h}h {m:02d}m" if h else f"{m}m"


def leg_fare(leg: dict) -> tuple[float | None, str | None]:
    for fp in leg["fareProducts"]:
        price = fp["product"].get("price")
        if price:
            return price["amount"], price["currency"]["code"]
    return None, None


# Flatten legs across all itineraries into a readable DataFrame
rows = []
for i, itin in enumerate(itineraries):
    for leg in itin["legs"]:
        fare_amount, fare_currency = leg_fare(leg)
        rows.append({
            "itinerary": i,
            "mode": leg["mode"],
            "route": leg["route"]["shortName"] if leg["route"] else None,
            "from": leg["from"]["name"],
            "to": leg["to"]["name"],
            "depart": leg["start"]["scheduledTime"],
            "arrive": leg["end"]["scheduledTime"],
            "duration": format_duration(leg["duration"]),
            "distance_km": round(leg["distance"] / 1000, 2) if leg["distance"] else None,
            "fare": fare_amount,
            "currency": fare_currency,
        })

legs_df = pd.DataFrame(rows)
legs_df["depart"] = pd.to_datetime(legs_df["depart"]).dt.strftime("%H:%M")
legs_df["arrive"] = pd.to_datetime(legs_df["arrive"]).dt.strftime("%H:%M")
legs_df

,itinerary,mode,route,from,to,depart,arrive,duration,distance_km,fare,currency
0,0,WALK,NaN,Origin,Destination,08:00,09:50,1h 51m,7.97,NaN,NaN
1,1,WALK,NaN,Origin,Melbourne Central Station,08:00,08:06,6m,0.35,NaN,NaN
2,1,RAIL,Lilydale,Melbourne Central Station,Southern Cross Station,08:06,08:10,4m,1.71,5.7,AUD
3,1,WALK,NaN,Southern Cross Station,Southern Cross Railway Station/Spencer St #122,08:10,08:15,5m,0.28,NaN,NaN
4,1,TRAM,96,Southern Cross Railway Station/Spencer St #122,Belford St/Acland St #139,08:18,08:41,23m,7.95,5.7,AUD
5,1,WALK,NaN,Belford St/Acland St #139,Destination,08:41,08:48,8m,0.57,NaN,NaN
6,2,WALK,NaN,Origin,Queen St/Bourke St #4,08:04,08:16,12m,0.74,NaN,NaN
7,2,TRAM,96,Queen St/Bourke St #4,Belford St/Acland St #139,08:16,08:47,31m,8.75,5.7,AUD
8,2,WALK,NaN,Belford St/Acland St #139,Destination,08:47,08:54,8m,0.57,NaN,NaN


In [4]:
# Aggregate each itinerary into a single "route" row: total cost, distance, time
route_rows = []
for i, itin in enumerate(itineraries):
    legs = itin["legs"]
    transit_legs = [leg for leg in legs if leg["route"] is not None]
    modes_used = list(dict.fromkeys(leg["mode"] for leg in transit_legs))  # unique, in order

    total_distance_km = sum(leg["distance"] for leg in legs if leg["distance"]) / 1000

    total_fare = 0.0
    currency = None
    for leg in legs:
        amount, cur = leg_fare(leg)
        if amount:
            total_fare += amount
            currency = currency or cur

    route_rows.append({
        "itinerary": i,
        "origin": legs[0]["from"]["name"],
        "destination": legs[-1]["to"]["name"],
        "depart": itin["start"],
        "arrive": itin["end"],
        "duration": format_duration(itin["duration"]),
        "transfers": itin["numberOfTransfers"],
        "modes": " -> ".join(modes_used) if modes_used else "WALK",
        "distance_km": round(total_distance_km, 2),
        "total_fare": round(total_fare, 2) if currency else None,
        "currency": currency,
    })

routes_df = pd.DataFrame(route_rows)
routes_df["depart"] = pd.to_datetime(routes_df["depart"]).dt.strftime("%Y-%m-%d %H:%M")
routes_df["arrive"] = pd.to_datetime(routes_df["arrive"]).dt.strftime("%Y-%m-%d %H:%M")
routes_df

,itinerary,origin,destination,depart,arrive,duration,transfers,modes,distance_km,total_fare,currency
0,0,Origin,Destination,2026-05-20 08:00,2026-05-20 09:50,1h 51m,0,WALK,7.97,NaN,NaN
1,1,Origin,Destination,2026-05-20 08:00,2026-05-20 08:48,48m,1,RAIL -> TRAM,10.86,11.4,AUD
2,2,Origin,Destination,2026-05-20 08:04,2026-05-20 08:54,50m,0,TRAM,10.05,5.7,AUD
